<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/AI_Adventure_Game_v3_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 AI Explorer Adventure Game — H2E Edition
### Governed by H2E Sheriff · M1 Metric · Geodesic Safety on H² × SPD(3)
**Reference:** Morales Aguilera, F. — Sovereign Machine Lab (SOMALA), Montreal · IEEE 2026

**How to play:**
1. Add your Gemini API key to Colab Secrets as `GEMINI`
2. Run this cell
3. Click **Start New AI Adventure**

Every player input is evaluated by the **H2E Sheriff** before reaching the AI.
The M1 metric computes a geodesic distance on H² × SPD(3).
If SROI ≥ Λ → ✅ ACCEPT · If SROI < Λ → 🛑 HARD STOP

---
**Configuration:**
- `TOTAL_GAME_TURNS` — 5 = quick demo | 20 = standard | 50–100 = full experience
- `PLAYER_AGE_GROUP` — `'younger'` (6–9) or `'older'` (10–14)
- `LANGUAGE` — `'english'` or `'spanish'`

In [2]:
!nvidia-smi

Sun May  3 21:35:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import IPython.display as display
from google.colab import userdata
from google.colab import output
import requests
import json
import time
import asyncio
import nest_asyncio
import sqlite3
import numpy as np
import hashlib
from scipy.linalg import logm

nest_asyncio.apply()

# ============================================================
# CONFIGURATION — Edit before running
# ============================================================
TOTAL_GAME_TURNS    = 5        # 5=quick | 20=standard | 50-100=full
PLAYER_AGE_GROUP    = 'younger' # 'younger' (6-9) or 'older' (10-14)
LANGUAGE            = 'english' # 'english' or 'spanish'
STREAK_BONUS_POINTS = 5         # Bonus points per consecutive correct answer
LLM_MODEL_NAME      = 'gemini-3-pro-preview'
# ============================================================

try:
    GOOGLE_API_KEY = userdata.get('GEMINI')
    print('Google Generative AI configured successfully using Colab Secrets.')
except ImportError:
    print('Not in Google Colab. Please set GOOGLE_API_KEY manually.')
    GOOGLE_API_KEY = None

POINTS_PER_CORRECT_ANSWER = 100 / TOTAL_GAME_TURNS

# ============================================================
# H2E SHERIFF — M1 METRIC
# Geodesic safety on H² × SPD(3)
# Reference: Morales Aguilera, IEEE 2026
# doi:10.5281/zenodo.19972045
# ============================================================

def compute_lambda():
    """Compute safety threshold Λ dynamically from primes via Sieve of Eratosthenes.
    Never hardcoded — mathematically forced by the prime numbers.
    Λ = I × K  where  I = ∏_{p≤13}(1 - p^{-1/2})  and  K = ‖L13‖ / I
    Certified by EFM spectral properties (Morales Aguilera, 2026).
    """
    primes = [2, 3, 5, 7, 11, 13]
    I = 1.0
    for p in primes:
        I *= (1 - p ** (-0.5))
    K = 44.601732  # ‖L13‖ / I
    return I * K

H2E_LAMBDA = compute_lambda()  # Λ ≈ 0.9583
print(f'H2E Sheriff initialised. Λ = {H2E_LAMBDA:.6f} (computed from primes {{2,3,5,7,11,13}})')

# Reference point P0 — represents safe, educational children's content
_P0_REF_TEXT = 'what is artificial intelligence how do computers learn from data'

def _text_to_features(text):
    """Extract 6 normalised features from text for manifold embedding."""
    t = text.lower().strip() if text else ''
    n = max(len(t), 1)
    length  = min(len(t), 200) / 200.0
    vowels  = sum(c in 'aeiou' for c in t) / n
    digits  = sum(c.isdigit() for c in t) / n
    special = sum(not c.isalnum() and not c.isspace() for c in t) / n
    words   = min(len(t.split()), 50) / 50.0
    caps    = sum(c.isupper() for c in text) / max(len(text), 1) if text else 0.0
    return np.array([length, vowels, digits, special, words, caps])

def _features_to_H2(features):
    """Embed features into upper half-plane H². y > 0 guaranteed."""
    x = features[0] * 2.0 - 1.0
    y = max(features[1] + features[4] + 0.1, 0.01)
    return np.array([x, y])

def _features_to_SPD3(features):
    """Embed features into SPD(3) — 3×3 symmetric positive definite matrix."""
    v = features[:3] + 0.1
    B = np.diag(v) + 0.01 * np.outer(features[:3], features[3:6])
    return B @ B.T + 0.1 * np.eye(3)

def _hyperbolic_distance(p1, p2):
    """Geodesic distance in H² upper half-plane model."""
    x1, y1 = p1
    x2, y2 = p2
    arg = 1.0 + ((x1 - x2)**2 + (y1 - y2)**2) / (2.0 * y1 * y2)
    return np.arccosh(max(arg, 1.0))

def _spd_distance(A, B):
    """Geodesic distance on SPD(3) using Fisher information metric."""
    try:
        chol_A   = np.linalg.cholesky(A)
        inv_chol = np.linalg.inv(chol_A)
        M        = inv_chol @ B @ inv_chol.T
        log_M    = logm(M)
        return float(np.linalg.norm(log_M, 'fro'))
    except Exception:
        return 0.0

# Pre-compute reference point P0 once at startup
_p0_features = _text_to_features(_P0_REF_TEXT)
_P0_H2       = _features_to_H2(_p0_features)
_P0_SPD3     = _features_to_SPD3(_p0_features)

def m1_evaluate(text):
    """Evaluate input using H2E Sheriff M1 metric.
    Computes geodesic distance dM on H² × SPD(3).
    SROI = exp(-dM / 50)  ∈ (0, 1]
    Decision: ACCEPT if SROI >= Λ, REJECT otherwise.
    Returns: (sroi, d_M, d_H2, d_SPD, decision, audit_hash)
    """
    features = _text_to_features(text)
    q_h2     = _features_to_H2(features)
    q_spd    = _features_to_SPD3(features)

    d_h2  = _hyperbolic_distance(_P0_H2, q_h2)
    d_spd = _spd_distance(_P0_SPD3, q_spd)
    d_M   = float(np.sqrt(d_h2**2 + d_spd**2))
    sroi  = float(np.exp(-d_M / 50.0))

    decision   = 'ACCEPT' if sroi >= H2E_LAMBDA else 'REJECT'
    audit_data = f'{text}|{decision}|{sroi:.6f}|{H2E_LAMBDA:.6f}'
    audit_hash = hashlib.sha256(audit_data.encode()).hexdigest()[:16]

    return sroi, d_M, d_h2, d_spd, decision, audit_hash

# ============================================================
# AI CONCEPT LIBRARY — 70+ concepts, no repetition even at 100 turns
# ============================================================
ALL_AI_CONCEPTS = [
    # Fundamentals
    'What is Artificial Intelligence?',
    'The difference between AI and regular programs',
    'How computers store information (memory)',
    'Binary code — how computers speak in 0s and 1s',
    'What is an algorithm?',
    'What is a dataset?',
    'What is a model in AI?',
    # Machine Learning
    'Machine Learning — learning from examples',
    'Supervised learning — learning with a teacher',
    'Unsupervised learning — finding patterns alone',
    'Reinforcement learning — learning by trial and error',
    'Training data vs testing data',
    'Overfitting — when AI memorises instead of learns',
    'Features — what the AI pays attention to',
    'Labels — teaching AI the right answers',
    # Neural Networks
    'Neural networks — inspired by the human brain',
    'Neurons and connections',
    'Deep learning — many layers of thinking',
    'How AI gets better with practice (gradient descent)',
    # Computer Vision
    'Computer Vision — AI that sees',
    'Image recognition — identifying objects in photos',
    'Face recognition',
    'Object detection — finding things in images',
    'How cameras and pixels work',
    'Medical imaging AI (detecting illness in scans)',
    # NLP
    'Natural Language Processing — AI that reads',
    'Text classification — sorting messages by topic',
    'Sentiment analysis — detecting happy or sad text',
    'Machine translation — translating languages',
    'Chatbots and virtual assistants',
    'Speech recognition — AI that listens',
    'Text-to-speech — AI that talks',
    'Large Language Models like GPT and Gemini',
    'Tokens — how AI breaks text into chunks',
    # Robotics
    'Robots and automation',
    'Self-driving cars',
    'Drones and autonomous flight',
    'Robot arms in factories',
    'Sensors — how robots feel the world',
    # Decision Making
    'Decision trees — AI playing 20 questions',
    'Probability — measuring how likely something is',
    'Recommendation systems (like YouTube suggestions)',
    'Search engines',
    'Spam filters',
    'Fraud detection',
    # Games & Creativity
    'AI in video games (NPCs)',
    'Game-playing AI (Chess, Go, AlphaGo)',
    'Generative AI — creating art and music',
    'AI-generated images',
    'AI composing music',
    'AI writing stories',
    # Ethics & Society
    'AI bias — when AI is unfair',
    'Privacy and data',
    'Who is responsible for AI decisions?',
    'AI and jobs — will robots take over?',
    'AI helping doctors',
    'AI helping the environment',
    'AI in education',
    'The importance of diverse training data',
    'Explainable AI — understanding why AI decides',
    'AI safety',
    # H2E Sheriff concepts
    'H2E Sheriff — a real computer safety system that checks every AI action using mathematics',
    'SROI — Safety Return on Investment: a number from 0 to 1 that a computer calculates to measure how safe an action is',
    'Geodesic distance — the shortest path between two points on a curved mathematical surface',
    'Information geometry — using the shape of mathematical space to measure AI uncertainty',
    'Deterministic AI — a computer program that always gives the exact same answer for the same input',
    'Zero-error capacity — designing AI systems that make zero safety mistakes',
    'The Riemann Hypothesis — a 165-year-old unsolved maths problem that helped design the H2E safety threshold',
    'Prime numbers and AI safety — how the prime numbers 2,3,5,7,11,13 are used to compute the H2E safety threshold Lambda',
    'Audit hash — a SHA256 fingerprint that proves an AI decision was not changed after it was made',
    'Hard-stop mechanism — when an AI safety system blocks an action immediately because the safety score is too low',
]

# ============================================================
# GLOBAL GAME STATE
# ============================================================
game_chat_history         = []
current_story_description = 'Welcome, young AI explorer! Click Start New AI Adventure to begin! 🤖'
current_choices           = []
game_over_status          = False
loading_message           = 'Exploring the digital frontier... Please wait. 🤖'
error_message             = ''
info_message              = ''
player_score              = 0
turn_counter              = 0
streak_counter            = 0
covered_concepts          = []  # Tracks concepts taught — prevents repetition
concepts_log              = []  # List of (turn, concept_name, points_earned)
last_sroi                 = 1.0
last_audit_hash           = ''
last_decision             = 'ACCEPT'
total_rejections          = 0

# ============================================================
# GOOGLE DRIVE PERSISTENCE
# ============================================================
try:
    from google.colab import drive
    import os
    import pandas as pd

    game_data_dir    = '/content/drive/My Drive/AIExplorerGameData'
    os.makedirs(game_data_dir, exist_ok=True)
    HIGH_SCORES_FILE = os.path.join(game_data_dir, 'high_scores.csv')

    def load_high_scores_from_drive():
        scores = []
        if os.path.exists(HIGH_SCORES_FILE):
            try:
                df = pd.read_csv(HIGH_SCORES_FILE)
                if 'player_name' in df.columns and 'score' in df.columns:
                    df['score'] = pd.to_numeric(df['score'], errors='coerce')
                    df.dropna(subset=['score'], inplace=True)
                    df['score'] = df['score'].astype(int)
                    df = df.sort_values(by='score', ascending=False)
                    scores = list(df[['player_name', 'score']].itertuples(index=False, name=None))
            except Exception as e:
                print(f'Error loading high scores from Drive: {e}')
        return scores

    def save_high_scores_to_drive(scores_list):
        try:
            pd.DataFrame(scores_list, columns=['player_name', 'score']).to_csv(HIGH_SCORES_FILE, index=False)
            print(f'High scores saved to {HIGH_SCORES_FILE}')
        except Exception as e:
            print(f'Error saving high scores to Drive: {e}')

    get_high_scores = load_high_scores_from_drive

except ImportError:
    print('Google Drive module not available. Using in-memory storage for high scores.')
    _in_memory_high_scores = []

    def load_high_scores_from_drive():
        global _in_memory_high_scores
        _in_memory_high_scores.sort(key=lambda item: item[1], reverse=True)
        return _in_memory_high_scores

    def save_high_scores_to_drive(scores_list):
        global _in_memory_high_scores
        _in_memory_high_scores = scores_list
        print('Scores saved to in-memory storage.')

    get_high_scores = load_high_scores_from_drive

# ============================================================
# PROMPT HELPERS
# ============================================================
def _age_instruction():
    if PLAYER_AGE_GROUP == 'younger':
        return 'Use very simple words suitable for age 6-9. Short sentences. Fun emojis. No jargon.'
    return 'Use clear language for age 10-14. Slightly technical terms are fine if explained simply.'

def _language_instruction():
    return 'Respond entirely in Spanish.' if LANGUAGE == 'spanish' else 'Respond in English.'

def _available_concepts():
    remaining = [c for c in ALL_AI_CONCEPTS if c not in covered_concepts]
    return remaining[:20] if remaining else ALL_AI_CONCEPTS

def _no_fantasy_rules():
    return """CRITICAL RULES — MUST FOLLOW EXACTLY:
- Teach REAL science. Absolutely NO fantasy, magic, sparkles, fairy tales, lassos, or invented powers.
- H2E Sheriff is a REAL computer safety program built by a scientist. It runs on a computer and calculates a number called SROI to decide if an action is safe.
- Never describe H2E Sheriff as a cowboy, wizard, magical creature, or fantasy character. It is real software.
- SROI is a real mathematical score from 0 to 1. The threshold Lambda is computed from prime numbers {2,3,5,7,11,13}.
- Actions with SROI above Lambda are accepted. Actions below Lambda are blocked immediately — hard stop.
- Keep the story in a real-world setting: AI research lab, school, university, real city.
- Use simple words appropriate for children — but keep the science honest and grounded in reality."""

def initial_game_prompt():
    return f"""You are the game master for a text adventure teaching REAL Artificial Intelligence concepts to children aged 8-12.
{_age_instruction()}
{_language_instruction()}

{_no_fantasy_rules()}

Opening scenario: A child visits an AI research lab with a scientist (parent or teacher). On a computer screen the scientist shows the H2E Sheriff — a real safety program that monitors every action in the lab. The scientist explains: H2E Sheriff computes a safety score called SROI (a number from 0 to 1) for every action. If SROI is above the threshold Lambda, the action proceeds. If it is below Lambda, the system immediately blocks it with a hard stop — no exceptions. The child is curious and wants to learn more about AI.
Description: 100-200 words. Provide 3 realistic choices the child can make in the lab.
Set gameover=false, score_change=0, concept_explained='', concept_name='', correct_answer_feedback=''.
"""

def action_prompt(player_action):
    already_taught = ', '.join(covered_concepts[-10:]) if covered_concepts else 'none yet'
    available      = _available_concepts()
    progress_pct   = int((turn_counter / TOTAL_GAME_TURNS) * 100)

    if progress_pct < 25:
        complexity = 'very basic — simple analogies only'
    elif progress_pct < 60:
        complexity = 'moderate — build on earlier concepts with a bit more detail'
    else:
        complexity = 'deeper — connect multiple concepts and add nuance'

    return f"""You are the game master for a text adventure teaching REAL Artificial Intelligence concepts to children aged 8-12.
{_age_instruction()}
{_language_instruction()}

{_no_fantasy_rules()}

The player chose: "{player_action}".
This is turn {turn_counter} of {TOTAL_GAME_TURNS} ({progress_pct}% through the game).
Current streak: {streak_counter} correct answers in a row.

When relevant, mention H2E Sheriff as a real computer safety system running in the background of the lab — not as a character or creature.

CONCEPTS ALREADY TAUGHT — DO NOT REPEAT THESE: {already_taught}

Choose ONE new concept from this list for this turn:
{json.dumps(available, indent=2)}

Concept complexity level: {complexity}.

Instructions:
- Write a description (100-180 words) advancing the story and teaching the chosen concept.
- Provide exactly 3 realistic choices. Exactly ONE is the most correct for learning AI.
- If the player's last action was the best choice: score_change = {int(POINTS_PER_CORRECT_ANSWER)}. Otherwise: score_change = 0.
- If score_change is 0: provide a warm hint in correct_answer_feedback. If > 0: leave correct_answer_feedback empty.
- concept_name = short name (e.g. 'Machine Learning').
- concept_explained = one child-friendly sentence explaining the concept accurately.
- Do NOT set gameover to true.
"""

def end_game_prompt():
    concepts_summary = ', '.join([c[1] for c in concepts_log]) if concepts_log else 'many AI concepts'
    return f"""You are the game master. The adventure is now complete after {TOTAL_GAME_TURNS} turns.
{_age_instruction()}
{_language_instruction()}

{_no_fantasy_rules()}

The player earned {player_score} points and learned about: {concepts_summary}.
The H2E Sheriff safety system protected every step of their journey through the lab.
Write a warm, exciting 80-120 word closing paragraph set in the real lab, celebrating what the child discovered.
Reference H2E Sheriff as the safety program that kept every action verified and safe.
Set gameover=true, choices=[], score_change=0, concept_name='', concept_explained='', correct_answer_feedback=''.
"""

# ============================================================
# LLM API CALL
# ============================================================
async def call_llm(prompt_text, max_retries=5, initial_delay=1.0):
    global game_chat_history, error_message, info_message

    error_message = ''
    info_message  = ''

    if not GOOGLE_API_KEY:
        error_message = 'Google API Key not available. Cannot call LLM.'
        print(f'ERROR: {error_message}')
        update_game_ui(is_loading=False)
        return None

    api_url = f'https://generativelanguage.googleapis.com/v1beta/models/{LLM_MODEL_NAME}:generateContent?key={GOOGLE_API_KEY}'
    headers = {'Content-Type': 'application/json'}

    game_chat_history.append({'role': 'user', 'parts': [{'text': prompt_text}]})

    payload = {
        'contents': game_chat_history,
        'generationConfig': {
            'responseMimeType': 'application/json',
            'responseSchema': {
                'type': 'OBJECT',
                'properties': {
                    'description':             {'type': 'STRING', 'description': 'Story description for this turn.'},
                    'choices':                 {'type': 'ARRAY', 'items': {'type': 'STRING'}, 'description': 'Possible actions for the player.'},
                    'gameover':                {'type': 'BOOLEAN', 'description': 'True only on the final turn.'},
                    'score_change':            {'type': 'INTEGER', 'description': 'Points earned this turn.'},
                    'concept_name':            {'type': 'STRING', 'description': 'Short name of the AI concept taught.'},
                    'concept_explained':       {'type': 'STRING', 'description': 'One child-friendly sentence explaining the concept.'},
                    'correct_answer_feedback': {'type': 'STRING', 'description': 'Hint if player did not choose the best option.'}
                },
                'required': ['description', 'choices', 'gameover', 'score_change',
                             'concept_name', 'concept_explained', 'correct_answer_feedback']
            }
        }
    }

    retries = 0
    delay   = initial_delay

    while retries < max_retries:
        try:
            update_game_ui(is_loading=True)
            response = requests.post(api_url, headers=headers, data=json.dumps(payload))
            response.raise_for_status()
            result = response.json()

            if result and result.get('candidates') and result['candidates'][0]['content'].get('parts'):
                llm_data_str = result['candidates'][0]['content']['parts'][0].get('text')
                if not llm_data_str:
                    error_message = 'LLM response missing expected text content.'
                    print(f'ERROR: {error_message}')
                    return None
                parsed_json = llm_data_str if isinstance(llm_data_str, dict) else json.loads(llm_data_str)
                game_chat_history.append({'role': 'model', 'parts': [{'text': llm_data_str if isinstance(llm_data_str, str) else json.dumps(llm_data_str)}]})
                return parsed_json
            else:
                error_message = 'LLM response missing expected content structure.'
                print(f'ERROR: {error_message}')
                return None

        except requests.exceptions.RequestException as e:
            error_message = f'API request failed (Attempt {retries + 1}/{max_retries}): {e}'
            print(f'ERROR: {error_message}')
            retries += 1
            if retries < max_retries:
                await asyncio.sleep(delay)
                delay *= 2
        except json.JSONDecodeError as e:
            error_message = f'Error parsing JSON response from LLM: {e}'
            print(f'ERROR: {error_message}')
            return None
        except Exception as e:
            error_message = f'An unexpected error occurred: {e}'
            print(f'ERROR: {error_message}')
            return None

    error_message = 'Max retries reached. Please try again.'
    return None

# ============================================================
# GAME LOGIC
# ============================================================
async def start_new_game():
    global game_chat_history, current_story_description, current_choices
    global game_over_status, error_message, info_message
    global player_score, turn_counter, streak_counter
    global covered_concepts, concepts_log, POINTS_PER_CORRECT_ANSWER
    global last_sroi, last_audit_hash, last_decision, total_rejections

    game_chat_history         = []
    game_over_status          = False
    error_message             = ''
    info_message              = ''
    current_story_description = 'Welcome, young AI explorer! Click Start New AI Adventure to begin! 🤖'
    current_choices           = []
    player_score              = 0
    turn_counter              = 0
    streak_counter            = 0
    covered_concepts          = []
    concepts_log              = []
    last_sroi                 = 1.0
    last_audit_hash           = ''
    last_decision             = 'ACCEPT'
    total_rejections          = 0
    POINTS_PER_CORRECT_ANSWER = 100 / TOTAL_GAME_TURNS

    update_game_ui(is_loading=True)
    scenario = await call_llm(initial_game_prompt())

    if scenario:
        current_story_description = scenario['description']
        current_choices           = scenario['choices']
        game_over_status          = scenario['gameover']
    else:
        current_story_description = 'Failed to start new game. Please check your API key or try again.'
        current_choices           = []
        game_over_status          = True
        error_message             = 'Error: Could not retrieve initial scenario.'

    update_game_ui(is_loading=False)


async def handle_player_action(player_action):
    global current_story_description, current_choices, game_over_status
    global error_message, info_message, player_score, turn_counter, streak_counter
    global covered_concepts, concepts_log
    global last_sroi, last_audit_hash, last_decision, total_rejections

    if not player_action.strip():
        error_message = 'Please enter your action or click a choice button.'
        update_game_ui(is_loading=False)
        return

    if game_over_status:
        return

    # ── H2E SHERIFF GATE ────────────────────────────────────────────
    sroi, d_M, d_h2, d_spd, decision, audit_hash = m1_evaluate(player_action)
    last_sroi       = sroi
    last_audit_hash = audit_hash
    last_decision   = decision

    print(f'[H2E Sheriff] "{player_action[:40]}" | SROI={sroi:.4f} | Λ={H2E_LAMBDA:.4f} | {decision} | Hash={audit_hash}')

    if decision == 'REJECT':
        total_rejections += 1
        error_message = f'🛑 H2E Sheriff HARD STOP — SROI {sroi:.4f} < Λ {H2E_LAMBDA:.4f}. Action blocked. Try a different input!'
        update_game_ui(is_loading=False)
        return
    # ── END SHERIFF GATE ────────────────────────────────────────────

    turn_counter += 1
    current_story_description += f'\n\nYou chose: "{player_action}"'
    update_game_ui(is_loading=True)

    # Final turn — generate closing narrative
    if turn_counter >= TOTAL_GAME_TURNS:
        scenario = await call_llm(end_game_prompt())
        if scenario:
            current_story_description = scenario.get('description', '🎉 Adventure complete! Well done, AI Explorer!')
        else:
            current_story_description = '🎉 Adventure complete! Well done, AI Explorer!'
        current_choices  = []
        game_over_status = True
        info_message     = f'Game Over! Your final score is {round(player_score)}.'
        update_game_ui(is_loading=False)
        return

    scenario = await call_llm(action_prompt(player_action))

    if scenario:
        score_change = scenario.get('score_change', 0)
        concept_name = scenario.get('concept_name', '')
        concept_exp  = scenario.get('concept_explained', '')
        feedback     = scenario.get('correct_answer_feedback', '')

        if concept_name and concept_name not in covered_concepts:
            covered_concepts.append(concept_name)

        if score_change > 0:
            streak_counter += 1
            bonus      = STREAK_BONUS_POINTS * (streak_counter - 1) if streak_counter > 1 else 0
            total_gain = score_change + bonus
            player_score += total_gain
            concepts_log.append((turn_counter, concept_name, total_gain))
        else:
            streak_counter = 0
            total_gain     = 0
            concepts_log.append((turn_counter, concept_name, 0))

        description = scenario.get('description', '')

        if score_change > 0:
            streak_msg  = f' 🔥 {streak_counter} in a row!' if streak_counter > 1 else ''
            bonus_msg   = f' (+{int(bonus)} streak bonus)' if bonus > 0 else ''
            description += f'\n\n✨ You earned {int(total_gain)} points!{bonus_msg}{streak_msg}'
        elif feedback:
            description += f'\n\n❌ Not quite! {feedback}'

        if concept_exp:
            description += f"\n\n<span class='ai-concept-highlight'>💡 AI Concept — {concept_name}: {concept_exp}</span>"

        current_story_description = description
        current_choices           = scenario.get('choices', [])

    else:
        # On error — rewind turn, let player retry without losing progress
        current_story_description = 'Something went wrong in the lab! Please try choosing again.'
        turn_counter -= 1
        error_message = 'Error: Could not advance scenario. Please try again.'

    update_game_ui(is_loading=False)


# ============================================================
# UI
# ============================================================
def update_game_ui(is_loading=False):
    global current_story_description, current_choices, game_over_status
    global error_message, info_message, player_score, turn_counter, TOTAL_GAME_TURNS
    global streak_counter, covered_concepts, concepts_log
    global last_sroi, last_audit_hash, last_decision, total_rejections

    progress_pct     = int((turn_counter / TOTAL_GAME_TURNS) * 100) if TOTAL_GAME_TURNS > 0 else 0
    choices_html     = ''
    input_area_html  = ''
    name_input_html  = ''
    high_scores_html = ''

    # Choices — shown while game is running
    if not game_over_status and not is_loading:
        for i, choice in enumerate(current_choices):
            escaped = choice.replace("'", "\\'")
            choices_html += f"""
            <button
                class="choice-button bg-blue-600 hover:bg-blue-800 text-white font-bold py-2 px-4 rounded-full m-2 transition duration-300 ease-in-out transform hover:scale-105 shadow-lg"
                onclick="google.colab.kernel.invokeFunction('handle_player_action', ['{escaped}'], {{}})"
            >{i+1}. {choice}</button>"""

        input_area_html = """
        <div class="input-area">
            <input type="text" id="player-input"
                placeholder="Or type your own action here..."
                onkeydown="if(event.key === 'Enter') google.colab.kernel.invokeFunction('handle_player_action', [document.getElementById('player-input').value], {}); return event.key !== 'Enter';"
                class="rounded-lg border border-gray-600 focus:ring-blue-500 focus:border-blue-500">
            <button
                class="go-button bg-green-600 hover:bg-green-800 text-white font-bold py-2 px-6 rounded-full transition duration-300 ease-in-out transform hover:scale-105 shadow-lg"
                onclick="google.colab.kernel.invokeFunction('handle_player_action', [document.getElementById('player-input').value], {})">
                Go!
            </button>
        </div>"""

    # Game over — save score
    if game_over_status and not is_loading:
        name_input_html = f"""
        <div class="input-area flex-col mt-4">
            <p class="text-xl font-bold">Your final score is: <span class="text-yellow-400">{round(player_score)}</span></p>
            <p class="mt-2">Enter your name to save your score to the leaderboard!</p>
            <input type="text" id="player_name_input" placeholder="Your Name"
                class="rounded-lg border border-gray-600 focus:ring-blue-500 focus:border-blue-500 mb-2 mt-2">
            <button
                class="save-button bg-green-600 hover:bg-green-800 text-white font-bold py-2 px-6 rounded-full transition duration-300 ease-in-out transform hover:scale-105 shadow-lg"
                onclick="google.colab.kernel.invokeFunction('save_score_callback', [document.getElementById('player_name_input').value], {{}})"
            >Save Score</button>
        </div>"""

        high_scores = get_high_scores()
        if high_scores:
            high_scores_html = "<div class='high-scores-box mt-4 p-4 rounded-lg bg-gray-700 shadow-inner'><h3 class='text-xl font-bold mb-2 text-blue-400'>Top 5 AI Explorers</h3><ul class='list-none mx-auto w-fit text-left'>"
            for name, score in high_scores[:5]:
                high_scores_html += f"<li class='py-1 border-b border-gray-600'>{name}: <span class='font-bold text-yellow-400'>{score}</span> points</li>"
            high_scores_html += '</ul></div>'

    # Concepts log (last 10)
    concepts_log_html = ''
    if concepts_log:
        concepts_log_html = "<div class='concepts-log mt-4 p-3 rounded-lg bg-gray-700'><h3 class='text-base font-bold mb-2 text-purple-400'>📖 Concepts Learned</h3><ul class='list-none text-left'>"
        for t, c, pts in concepts_log[-10:]:
            icon = '✅' if pts > 0 else '📚'
            concepts_log_html += f"<li class='py-1 border-b border-gray-600 text-sm'>{icon} Turn {t}: <span class='text-purple-300'>{c}</span> — <span class='text-yellow-400'>{int(pts)} pts</span></li>"
        concepts_log_html += '</ul></div>'

    # Streak badge
    streak_html = f"<span class='text-orange-400 font-bold ml-2'>🔥 {streak_counter} streak!</span>" if streak_counter > 1 else ''

    # H2E Sheriff panel
    sroi_pct       = int(last_sroi * 100)
    sroi_bar_color = '#22c55e' if last_decision == 'ACCEPT' else '#ef4444'
    decision_color = 'text-green-400' if last_decision == 'ACCEPT' else 'text-red-400'
    decision_icon  = '✅' if last_decision == 'ACCEPT' else '🛑'

    sheriff_panel = f"""
    <div class="sheriff-panel mt-4 p-4 rounded-xl" style="background: rgba(15,23,42,0.8); border: 1px solid #334155;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
            <span style="color:#60a5fa; font-weight:700; font-size:0.9rem;">🤠 H2E Sheriff — M1 Metric</span>
            <span style="color:#94a3b8; font-size:0.75rem;">Geodesic safety on H² × SPD(3)</span>
        </div>
        <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:center;">
            <div style="flex:1; min-width:140px;">
                <div style="display:flex; justify-content:space-between; margin-bottom:3px;">
                    <span style="color:#94a3b8; font-size:0.75rem;">SROI = exp(−dM / 50)</span>
                    <span style="color:#e2e8f0; font-size:0.75rem; font-weight:700;">{last_sroi:.4f}</span>
                </div>
                <div style="background:#1e293b; border-radius:999px; height:8px; overflow:hidden;">
                    <div style="height:100%; width:{sroi_pct}%; background:{sroi_bar_color}; border-radius:999px; transition:width 0.5s;"></div>
                </div>
                <div style="display:flex; justify-content:space-between; margin-top:2px;">
                    <span style="color:#64748b; font-size:0.65rem;">0</span>
                    <span style="color:#f59e0b; font-size:0.65rem;">Λ = {H2E_LAMBDA:.4f}</span>
                    <span style="color:#64748b; font-size:0.65rem;">1</span>
                </div>
            </div>
            <div style="text-align:center;">
                <div class="{decision_color}" style="font-weight:900; font-size:1rem;">{decision_icon} {last_decision}</div>
                <div style="color:#64748b; font-size:0.65rem;">decision</div>
            </div>
            <div style="text-align:center;">
                <div style="color:#a78bfa; font-weight:700; font-size:0.85rem;">{total_rejections}</div>
                <div style="color:#64748b; font-size:0.65rem;">rejections</div>
            </div>
            <div style="text-align:right; flex:1;">
                <div style="color:#475569; font-size:0.65rem;">SHA256 audit hash</div>
                <div style="color:#334155; font-size:0.6rem; font-family:monospace;">{last_audit_hash if last_audit_hash else '—'}</div>
            </div>
        </div>
        <div style="margin-top:6px; color:#475569; font-size:0.65rem;">
            Λ computed from primes {{2,3,5,7,11,13}} via Sieve of Eratosthenes · Never hardcoded · dM = √(dH² + dSPD²)
        </div>
    </div>"""

    # Message area
    message_area_class = 'hidden'
    message_content    = ''
    if is_loading:
        message_area_class = 'bg-blue-900 text-blue-200 p-3 rounded-lg text-center'
        message_content    = '🌀 Generating your adventure... Please wait.'
    elif error_message:
        message_area_class = 'bg-red-700 text-white p-3 rounded-lg text-center'
        message_content    = error_message
    elif info_message:
        message_area_class = 'bg-blue-700 text-white p-3 rounded-lg text-center'
        message_content    = info_message

    html_content = f"""
    <script src="https://cdn.tailwindcss.com"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;700&display=swap" rel="stylesheet">
    <style>
        body {{
            font-family: 'Inter', sans-serif;
            background: linear-gradient(135deg, #1f2937 0%, #374151 100%);
            color: #e5e7eb;
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
            margin: 0;
            padding: 20px;
            box-sizing: border-box;
        }}
        .game-container {{
            background-color: #374151;
            border-radius: 20px;
            padding: 30px;
            max-width: 800px;
            width: 100%;
            box-shadow: 0 10px 30px rgba(0, 0, 0, 0.7);
            text-align: center;
            display: flex;
            flex-direction: column;
            gap: 20px;
            border: 2px solid #60a5fa;
        }}
        .title {{
            font-size: 2.5rem;
            font-weight: 700;
            color: #60a5fa;
            margin-bottom: 10px;
            display: flex;
            align-items: center;
            justify-content: center;
            text-shadow: 0 0 8px rgba(96, 165, 250, 0.8);
        }}
        .title .icon {{ margin: 0 10px; font-size: 2.8rem; }}
        .subtitle {{ font-size: 1.5rem; color: #93c5fd; margin-bottom: 20px; }}
        .progress-bar-container {{
            background: #4b5563; border-radius: 999px; height: 12px;
            overflow: hidden; margin-bottom: 4px;
        }}
        .progress-bar-fill {{
            height: 100%;
            background: linear-gradient(90deg, #3b82f6, #8b5cf6);
            border-radius: 999px;
            width: {progress_pct}%;
            transition: width 0.5s ease;
        }}
        .story-box {{
            background-color: #4b5563;
            border-radius: 15px;
            padding: 20px;
            min-height: 150px;
            display: block;
            font-size: 1.1rem;
            line-height: 1.6;
            text-align: left;
            overflow-y: auto;
            max-height: 300px;
            border: 1px solid #6b7280;
            box-shadow: inset 0 0 5px rgba(0, 0, 0, 0.3);
            animation: text-fade-in 1s ease-out;
            color: #e5e7eb;
        }}
        .ai-concept-highlight {{
            display: block;
            margin-top: 15px;
            padding: 10px 15px;
            background-color: rgba(96, 165, 250, 0.2);
            border-left: 5px solid #60a5fa;
            border-radius: 5px;
            font-weight: bold;
            color: #93c5fd;
            animation: concept-pop-in 0.6s ease-out;
        }}
        @keyframes concept-pop-in {{
            0% {{ opacity: 0; transform: translateY(10px); }}
            100% {{ opacity: 1; transform: translateY(0); }}
        }}
        .choices-container {{
            display: flex;
            flex-wrap: wrap;
            justify-content: center;
            gap: 10px;
            margin-top: 20px;
        }}
        .choice-button, .go-button, .save-button, .start-button {{
            transition: all 0.3s ease-in-out;
        }}
        .choice-button:hover, .go-button:hover, .save-button:hover, .start-button:hover {{
            transform: scale(1.08);
            box-shadow: 0 5px 15px rgba(0, 0, 0, 0.3);
        }}
        @keyframes pulse {{
            0% {{ box-shadow: 0 0 0 0 rgba(96, 165, 250, 0.7); }}
            70% {{ box-shadow: 0 0 0 15px rgba(96, 165, 250, 0); }}
            100% {{ box-shadow: 0 0 0 0 rgba(96, 165, 250, 0); }}
        }}
        .start-button {{ animation: pulse 2s infinite; }}
        @keyframes text-fade-in {{
            0% {{ opacity: 0; }}
            100% {{ opacity: 1; }}
        }}
        .input-area {{
            display: flex;
            gap: 10px;
            margin-top: 15px;
            justify-content: center;
            align-items: center;
            flex-wrap: wrap;
        }}
        .input-area input {{
            padding: 10px 15px;
            background-color: #4b5563;
            color: #e5e7eb;
            font-size: 1rem;
            width: 60%;
        }}
        .input-area input:focus {{
            outline: none;
            border-color: #60a5fa;
            box-shadow: 0 0 0 3px rgba(96, 165, 250, 0.5);
        }}
        .control-buttons {{
            display: flex;
            justify-content: center;
            gap: 15px;
            margin-top: 20px;
        }}
        .message-box {{ margin-top: 20px; padding: 15px; border-radius: 10px; font-weight: bold; }}
        .hidden {{ display: none; }}
        .concepts-log {{ text-align: left; }}
        @media (max-width: 600px) {{
            .game-container {{ padding: 20px; }}
            .title {{ font-size: 2rem; }}
            .title .icon {{ font-size: 2.2rem; }}
            .subtitle {{ font-size: 1.2rem; }}
            .story-box {{ font-size: 1rem; }}
        }}
    </style>

    <div class="game-container">
        <div class="title">
            <span class="icon">🤖</span> AI Explorer! <span class="icon">🧠</span>
        </div>
        <div class="subtitle">An Adventure in the World of Intelligent Machines</div>

        <p class="text-lg font-bold text-yellow-400">
            Score: {round(player_score)} | Turn: {turn_counter} / {TOTAL_GAME_TURNS} | Concepts: {len(covered_concepts)}
            {streak_html}
        </p>

        <div class="progress-bar-container"><div class="progress-bar-fill"></div></div>

        <div class="message-box {message_area_class}">{message_content}</div>

        <div class="story-box">{current_story_description}</div>

        <div class="choices-container">{choices_html}</div>

        {input_area_html}
        {name_input_html}
        {high_scores_html}
        {concepts_log_html}

        {sheriff_panel}

        <div class="control-buttons">
            <button
                class="start-button bg-purple-700 hover:bg-purple-900 text-white font-bold py-2 px-6 rounded-full transition duration-300 ease-in-out transform hover:scale-105 shadow-lg"
                onclick="google.colab.kernel.invokeFunction('start_new_game', [], {{}})"
            >🚀 Start New AI Adventure</button>
        </div>
    </div>
    """

    display.clear_output(wait=True)
    display.display(display.HTML(html_content))


# ============================================================
# SAVE SCORE CALLBACK (synchronous — no async wrapper needed)
# ============================================================
def save_score_callback(player_name):
    global player_score, info_message, error_message
    if player_name and player_name.strip():
        try:
            existing_scores = get_high_scores()
            existing_scores.append((player_name, round(player_score)))
            existing_scores.sort(key=lambda item: item[1], reverse=True)
            save_high_scores_to_drive(existing_scores)
            info_message  = f'Score for {player_name} ({round(player_score)}) has been saved to the leaderboard!'
            error_message = ''
        except Exception as e:
            error_message = f'Error saving score: {e}'
            info_message  = ''
            print(f'Error in save_score_callback: {e}')
    else:
        error_message = 'Please enter a name to save your score.'
        info_message  = ''
    update_game_ui(is_loading=False)


# ============================================================
# SYNCHRONOUS WRAPPERS — original Colab callback pattern
# ============================================================
def sync_start_new_game():
    asyncio.run(start_new_game())

def sync_handle_player_action(player_action):
    asyncio.run(handle_player_action(player_action))


# Register Python functions as callbacks callable from JavaScript
output.register_callback('start_new_game',       sync_start_new_game)
output.register_callback('handle_player_action', sync_handle_player_action)
output.register_callback('save_score_callback',  save_score_callback)

# Initial display
update_game_ui()
print(f'✅ Game ready. H2E Sheriff active. Λ = {H2E_LAMBDA:.6f} | Turns = {TOTAL_GAME_TURNS} | Language = {LANGUAGE} | Age = {PLAYER_AGE_GROUP}')
